# 05 · Búsqueda y Filtros Vectoriales

Este notebook explora el motor de búsqueda vectorial de Aurum Market y demuestra:

- Búsquedas semánticas sin filtros.
- Búsquedas con filtros nativos en Qdrant (marca, color, active).
- Comparación entre resultados filtrados y no filtrados.
- Interpretación de resultados.
- Justificación técnica del uso de filtros vectoriales.

Este notebook utiliza:
- El índice vectorial construido en el notebook 04.
- El modelo E5-small.
- La colección Qdrant `aurum_e5_small`.


#### Importar librerías

In [1]:
import sys
import os

ROOT_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
if ROOT_DIR not in sys.path:
    sys.path.append(ROOT_DIR)

print("Ruta añadida al PYTHONPATH:", ROOT_DIR)


Ruta añadida al PYTHONPATH: /home/alexd/modulo_vector_bbdd/actividad_evaluable


In [2]:
import pandas as pd

from src.search_engine import search
from src.filters import make_filter, make_brand_filter, make_color_filter, make_active_filter
from src.utils import safe_read_csv, log_section


#### Cargar catálogo de muestra

In [3]:
log_section("Cargar catálogo de muestra")

df_catalog = safe_read_csv("../data/catalogo_muestra.csv")
df_catalog.head()


[AURUM] 
[AURUM] ============================================================
[AURUM] Cargar catálogo de muestra
[AURUM] ============================================================
[AURUM] [CSV] Cargado: ../data/catalogo_muestra.csv (1500 filas)


,record_id,product_id,title,brand,color,locale,text,catalog_version,active
0,000bd6e8-a995-56d0-ba03-559885ccef39,B0818K237B,Kanlin1986 Vestido Largo De Navidad para Mujer...,KanLin1986-Ropa,Negro,es,Kanlin1986 Vestido Largo De Navidad para Mujer...,1,True
1,0037a9df-8492-508f-8167-c09624801216,B086YX9RK5,IQOS Kit Iqos 3 Duo Blue Opk 1 200 g,IQOS,NaN,es,IQOS Kit Iqos 3 Duo Blue Opk 1 200 g. Marca: I...,1,True
2,003a8544-5ab1-5c8e-8fa5-612989e4a7a8,B07FRXCFJ1,ELINKUME Lámpara de pie regulable LED Lámpara ...,ELINKUME,Lámparas de Pie-espiral Estilo-regulable Led,es,ELINKUME Lámpara de pie regulable LED Lámpara ...,1,True
3,0058d463-2a7c-592c-b73c-ea7462fc566b,B0869L7SSX,Guillermo Lenteja Beluga Caviar Negra Salamanc...,Guillermo,NaN,es,Guillermo Lenteja Beluga Caviar Negra Salamanc...,1,True
4,0095c6f6-248f-52a1-87f3-cf0b537507bc,B0831CX4LK,COSTWAY Mesa de Ordenador Escritorio Esquina e...,COSTWAY,Marrón,es,COSTWAY Mesa de Ordenador Escritorio Esquina e...,1,True


#### Búsqueda vectorial sin filtros

In [4]:
log_section("Búsqueda sin filtros")

query = "zapatillas running hombre"
results_no_filter = search(query, top_k=10, model_name="e5_small")

pd.DataFrame(results_no_filter)


[AURUM] 
[AURUM] ============================================================
[AURUM] Búsqueda sin filtros
[AURUM] ============================================================


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[AURUM] [EMBEDDINGS] Modelo cargado: e5_small (intfloat/multilingual-e5-small)


,rank,record_id,product_id,title,brand,color,active,score
0,1,82f03147-afbe-5eee-98d0-1410b7eb2e9b,B08PSSSJC6,Camper Zapatillas Runner Up 40,Camper,Blanco,1,0.862515
1,2,bb686570-ebe4-5a8b-9b43-2ce1221db102,B095PN46NB,"ASICS Gel-Trabuco 9, Zapatillas para Correr Ho...",ASICS,Carrier Grey Electric Red,1,0.860843
2,3,d0196af3-f1a8-5182-b195-242ee573c04c,B07V8W6V3C,"Tommy Hilfiger Corporate Leather Sneaker, Zapa...",Tommy Hilfiger,Blanco White Ybs,1,0.854788
3,4,5f0bdcce-605f-5fbe-9703-06b147f66c8c,B07CZDMRXC,"PUMA Icra Trainer SD, Zapatillas, para Unisex ...",PUMA,Azul Peacoat Puma White,1,0.840005
4,5,3a20db48-18d1-58c7-8b9a-c1d5df8a3d22,B09CTGK5Q3,Conjunto de Chándal Hombre Completo Sudadera s...,FeelFree+,Caqui,1,0.839215
5,6,6237f6ec-7ce7-5c53-a006-e74924f82dde,B078HXM454,"Nike Zoom Stefan Janoski, Zapatillas Hombre, N...",NIKE,Negro Black 333824 067,1,0.835228
6,7,3f3da1d1-bffc-5ea0-a324-0b9436b0eaa0,B07XTK93KM,Great Bikers Gear – Pantalones vaqueros de ing...,GREAT BIKERS GEAR,Azul Claro,1,0.826622
7,8,9edcd67e-3358-5261-9c8d-68f4179a4098,B07DRK7CGM,FM London - Camisetas para Hombre con Tecnolog...,FM London,Multicolor (Assorted ),1,0.825035
8,9,623137af-8cbb-5d4b-98da-076000f659c2,B01N8Y14IY,VeloChampion Cubrezapatillas Impermeables VC C...,VeloChampion,Negro,1,0.824719
9,10,b631f099-5137-5c09-bb20-c18093b670aa,B07HHX8TNL,tianxinxishop Disfraz de Jon Snow para Hombre ...,tianxinxishop,Version 2,1,0.824464


#### Interpretación de resultados sin filtros

La búsqueda vectorial devuelve productos semánticamente relacionados:

- Zapatillas de running.
- Calzado deportivo.
- Modelos de hombre.
- Variantes de color y marca coherentes.

Observaciones:
- El motor vectorial captura sinónimos y variaciones de lenguaje.
- Los resultados son relevantes incluso sin filtros.
- La semántica supera claramente al baseline BM25.


#### Búsqueda con filtro por marca

In [5]:
log_section("Filtro por marca: NIKE")

brand_filter = make_brand_filter("NIKE")
results_brand = search(query, top_k=10, model_name="e5_small", filters=brand_filter)

pd.DataFrame(results_brand)


[AURUM] 
[AURUM] ============================================================
[AURUM] Filtro por marca: NIKE
[AURUM] ============================================================


,rank,record_id,product_id,title,brand,color,active,score
0,1,6237f6ec-7ce7-5c53-a006-e74924f82dde,B078HXM454,"Nike Zoom Stefan Janoski, Zapatillas Hombre, N...",NIKE,Negro Black 333824 067,1,0.835228
1,2,e1a0e559-6a49-5be5-b617-ec8a4899e975,B000G3T55M,NIKE Legasee Legging Swoosh Pantalones Deporti...,NIKE,Negro (Black/White 011),1,0.797890


#### Interpretación del filtro por marca

El filtro por marca restringe los resultados a productos NIKE.

Observaciones:
- La relevancia semántica se mantiene.
- Todos los resultados pertenecen a la marca solicitada.
- Qdrant aplica el filtro **dentro del motor vectorial**, no después.
- Esto es más eficiente y más preciso que filtrar post-búsqueda.

Conclusión:
**Los filtros vectoriales permiten búsquedas semánticas dentro de un subset del catálogo.**


#### Búsqueda con filtro por color

In [ ]:
log_section("Filtro por color: NEGRO")

color_filter = make_color_filter("Negro")
results_color = search(query, top_k=10, model_name="e5_small", filters=color_filter)

pd.DataFrame(results_color)


[AURUM] 
[AURUM] ============================================================
[AURUM] Filtro por color: BLACK
[AURUM] ============================================================


,rank,record_id,product_id,title,brand,color,active,score
0,1,623137af-8cbb-5d4b-98da-076000f659c2,B01N8Y14IY,VeloChampion Cubrezapatillas Impermeables VC C...,VeloChampion,Negro,1,0.824719
1,2,c1432ee1-2368-523f-977a-c8bae7fc4cb5,B07FK85D1N,Garmin 010-12740-00 Quickfit 22 Correa de relo...,Garmin,Negro,1,0.805242
2,3,49d8a71f-e57e-5bd4-aadf-cbb9c5611238,B07G434RZM,gracosy Botas de Mujer 2021 Otoño Invierno Gom...,gracosy,Negro,1,0.805040
3,4,bedba603-4c62-57b5-b712-c1b67e806787,B07G4376VK,gracosy Botas de Mujer 2021 Otoño Invierno Gom...,gracosy,Negro,1,0.803891
4,5,8b9827c9-19c2-520a-89a5-947cbc0d87ba,B08SMSZPNC,"Leggings deportivos para mujer, cintura alta, ...",Holly's by Godecom,Negro,1,0.801966
5,6,d5bc53a7-391e-59a8-b476-6a348211bd1c,B08XWGZNWB,Bricoferr PT600010 Par De Ruedas Metálicas 100...,Bricoferr,Negro,1,0.801582
6,7,d315b036-cc8a-5c51-8de1-3e5f44c66015,B07GJNCV8R,"Botines De Altos Tacón Mujer,Piel con Platafor...",JOYTO,Negro,1,0.800817
7,8,5bb8cf30-8794-5df3-99a0-875634226e5d,B002VEC3JE,Honeywell 1010970 Howard Leight Thunder T3 Ear...,Bilsom,Negro,1,0.800148
8,9,7f56218d-ac98-57d3-8fe0-cf034fe07c50,B078GWWSQ8,"Cressi Patrol Chaleco, Unisex Adulto, Negro, M",Cressi,Negro,1,0.799240
9,10,c47dbf18-06d8-50d8-95e0-13269710e050,B00A9DJDKQ,made4bikers: Bolsas interiores adecuado para d...,made4bikers,Negro,1,0.798937


#### Interpretación del filtro por color

El filtro por color funciona correctamente y restringe la búsqueda a productos cuyo valor exacto en el campo color es "Negro".

Observaciones:

- La búsqueda devuelve únicamente productos con color = "Negro", tal como aparece en el catálogo de muestra.
- La relevancia semántica se mantiene dentro del subset filtrado: los resultados siguen siendo coherentes con la consulta original.
- Valores como "BLACK", "Black/White 011" o "Negro Black 333824 067" no aparecen, porque el filtro utiliza coincidencia exacta (MatchValue) y no coincidencias parciales.
- Qdrant aplica el filtro antes de la búsqueda vectorial, reduciendo el espacio vectorial únicamente a los productos que cumplen el criterio.
- Esto permite búsquedas semánticas dentro de subsets muy específicos del catálogo, manteniendo eficiencia y precisión.

Conclusión:  
**Los filtros vectoriales permiten combinar semántica y estructura, realizando búsquedas relevantes dentro de un subconjunto exacto del catálogo.**


#### Búsqueda con filtro por estado activo

In [9]:
log_section("Filtro por estado activo")

active_filter = make_active_filter(1)
results_active = search(query, top_k=10, model_name="e5_small", filters=active_filter)

pd.DataFrame(results_active)


[AURUM] 
[AURUM] ============================================================
[AURUM] Filtro por estado activo
[AURUM] ============================================================


,rank,record_id,product_id,title,brand,color,active,score
0,1,82f03147-afbe-5eee-98d0-1410b7eb2e9b,B08PSSSJC6,Camper Zapatillas Runner Up 40,Camper,Blanco,1,0.862515
1,2,bb686570-ebe4-5a8b-9b43-2ce1221db102,B095PN46NB,"ASICS Gel-Trabuco 9, Zapatillas para Correr Ho...",ASICS,Carrier Grey Electric Red,1,0.860843
2,3,d0196af3-f1a8-5182-b195-242ee573c04c,B07V8W6V3C,"Tommy Hilfiger Corporate Leather Sneaker, Zapa...",Tommy Hilfiger,Blanco White Ybs,1,0.854788
3,4,5f0bdcce-605f-5fbe-9703-06b147f66c8c,B07CZDMRXC,"PUMA Icra Trainer SD, Zapatillas, para Unisex ...",PUMA,Azul Peacoat Puma White,1,0.840005
4,5,3a20db48-18d1-58c7-8b9a-c1d5df8a3d22,B09CTGK5Q3,Conjunto de Chándal Hombre Completo Sudadera s...,FeelFree+,Caqui,1,0.839215
5,6,6237f6ec-7ce7-5c53-a006-e74924f82dde,B078HXM454,"Nike Zoom Stefan Janoski, Zapatillas Hombre, N...",NIKE,Negro Black 333824 067,1,0.835228
6,7,3f3da1d1-bffc-5ea0-a324-0b9436b0eaa0,B07XTK93KM,Great Bikers Gear – Pantalones vaqueros de ing...,GREAT BIKERS GEAR,Azul Claro,1,0.826622
7,8,9edcd67e-3358-5261-9c8d-68f4179a4098,B07DRK7CGM,FM London - Camisetas para Hombre con Tecnolog...,FM London,Multicolor (Assorted ),1,0.825035
8,9,623137af-8cbb-5d4b-98da-076000f659c2,B01N8Y14IY,VeloChampion Cubrezapatillas Impermeables VC C...,VeloChampion,Negro,1,0.824719
9,10,b631f099-5137-5c09-bb20-c18093b670aa,B07HHX8TNL,tianxinxishop Disfraz de Jon Snow para Hombre ...,tianxinxishop,Version 2,1,0.824464


#### Interpretación del filtro active

El filtro `active=1` funciona correctamente y restringe la búsqueda a productos marcados como activos en el catálogo.

Observaciones:

- En el catálogo de muestra, todos los productos tienen active = 1, por lo que el filtro no modifica el conjunto de resultados.
- La búsqueda filtrada es idéntica a la búsqueda sin filtros, lo que confirma que el filtro se aplica correctamente.
- Qdrant ejecuta el filtro antes de calcular la similitud vectorial, garantizando que la búsqueda se realiza únicamente dentro del subset de productos activos.
- En un catálogo real, este filtro es esencial para evitar mostrar productos descatalogados, ocultos o eliminados.

Conclusión:  
**El filtro active es obligatorio en búsquedas reales, aunque en el catálogo de muestra no produce cambios porque todos los productos están activos.**


#### Comparación entre búsquedas filtradas y no filtradas

In [11]:
log_section("Comparación filtrado vs no filtrado")

# Obtener longitudes
max_len = max(
    len(results_no_filter),
    len(results_brand),
    len(results_color),
    len(results_active)
)

def pad(lst, target_len):
    return lst + [None] * (target_len - len(lst))

df_compare = pd.DataFrame({
    "sin_filtro": pad([r["product_id"] for r in results_no_filter], max_len),
    "marca_NIKE": pad([r["product_id"] for r in results_brand], max_len),
    "color_BLACK": pad([r["product_id"] for r in results_color], max_len),
    "active_1": pad([r["product_id"] for r in results_active], max_len),
})

df_compare


[AURUM] 
[AURUM] ============================================================
[AURUM] Comparación filtrado vs no filtrado
[AURUM] ============================================================


,sin_filtro,marca_NIKE,color_BLACK,active_1
0,B08PSSSJC6,B078HXM454,B01N8Y14IY,B08PSSSJC6
1,B095PN46NB,B000G3T55M,B07FK85D1N,B095PN46NB
2,B07V8W6V3C,NaN,B07G434RZM,B07V8W6V3C
3,B07CZDMRXC,NaN,B07G4376VK,B07CZDMRXC
4,B09CTGK5Q3,NaN,B08SMSZPNC,B09CTGK5Q3
5,B078HXM454,NaN,B08XWGZNWB,B078HXM454
6,B07XTK93KM,NaN,B07GJNCV8R,B07XTK93KM
7,B07DRK7CGM,NaN,B002VEC3JE,B07DRK7CGM
8,B01N8Y14IY,NaN,B078GWWSQ8,B01N8Y14IY
9,B07HHX8TNL,NaN,B00A9DJDKQ,B07HHX8TNL


### Conclusiones

- La búsqueda vectorial sin filtros ofrece resultados semánticamente sólidos y coherentes con la consulta.
- Los filtros nativos de Qdrant funcionan correctamente y permiten:
  - restringir por marca (en este dataset solo dos productos NIKE),
  - restringir por color (solo con coincidencia exacta del campo color),
  - restringir por estado activo (sin cambios en este catálogo porque todos los productos están activos).
- Los resultados filtrados mantienen la relevancia dentro del subset seleccionado.
- Qdrant aplica los filtros antes de calcular la similitud vectorial, lo que garantiza eficiencia y precisión.
- Este notebook demuestra que el sistema soporta búsquedas avanzadas combinando semántica y filtros estructurados.

En el siguiente notebook aplicaremos **eventos del catálogo** y analizaremos cómo afectan a la visibilidad de los productos.
